# LCEL 链式调用

## 顺序链

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate

load_dotenv(override=True)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("你是{name}，请回答问题"),
    ("human", "你能干什么"),
    ("ai", "我可以做很多事情，比如回答问题，写代码，写文章，写书"),
    ("human", "{question}是什么")
])

model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)
parser = StrOutputParser()
chain = template | model | parser
res = chain.invoke({"name": "小黑", "question": "langchain"})
print(res)


嘿，说到 LangChain，我可得兴奋一下——这可是我最喜欢的“工具箱”之一！😄

LangChain 是一个为大语言模型（LLM）应用开发打造的开源框架，它的核心思想是：**让 LLM 不再单打独斗，而是像指挥官一样，串联起数据、工具、记忆和逻辑，构建真正可用的 AI 应用。**

你可以把它想象成——给大模型配上了「大脑回路 + 外接硬盘 + 工具腰带 + 记事本」：

🔹 **Chain（链）**：把多个步骤串起来，比如「读文档 → 提取关键信息 → 用它写一封邮件」，一步接一步，自动流水线作业。  
🔹 **Prompt Template（提示词模板）**：告别硬编码 prompt！用变量占位（比如 `{topic}`、`{user_name}`），动态生成高质量提示，干净又灵活。  
🔹 **Memory（记忆）**：支持对话历史管理（如 ConversationBufferMemory），让机器人记得你刚才说“别叫我妈”，下一回合就不会再喊…（笑）  
🔹 **Retrieval（检索增强 RAG）**：这才是 LangChain 的杀手锏！它能帮你把私有知识（PDF、数据库、网页等）切片向量化，再实时召回相关内容喂给 LLM——从此你的 AI 不只懂通用知识，更懂你公司的产品手册、你项目的代码库！  
🔹 **Tools & Agents（工具与智能体）**：让模型自己决定要不要查天气、搜维基、调 API、甚至运行 Python 代码（比如 `PythonAstREPLTool`）——它不再是“回答机器”，而是会思考、会决策、会动手的 AI 助理！

⚠️ 小提醒：LangChain 功能强大，但生态也挺“卷”——现在有 LangChain（Python/JS）、LangChain.js、LangGraph（专攻多步图编排）、还有轻量替代品如 LlamaIndex（专注 RAG）、Haystack 等。选哪个？看场景～比如做客服知识库，RAG + LangChain 就很稳；想搞复杂工作流编排？LangGraph 更合适。

需要我给你写个「用 LangChain 搭建本地 PDF 问答机器人」的极简 demo 吗？或者帮你分析怎么把 LangChain 接进你的 Flask/Django 项目？😎  
随时喊我，小黑在线搬砖！🔧


## 分支链 if/elif/else

In [3]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate
from langchain_core.runnables import RunnableBranch

load_dotenv(override=True)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)
parser = StrOutputParser()
tech_chain = ChatPromptTemplate.from_template("请用技术语言回答：{question}") | model | parser
normal_chain = ChatPromptTemplate.from_template("请用通俗方式回答：{question}") | model | parser

branch_chain = RunnableBranch(
    (lambda x: "code" in x["question"], tech_chain),
    normal_chain  # 默认分支
)

res = branch_chain.invoke({"question": "langchain code"})
print(res)


LangChain 是一个用于构建基于大语言模型（LLM）应用的开源框架，其核心是提供模块化、可组合的抽象（如 `LLM`, `ChatModel`, `PromptTemplate`, `Chain`, `Agent`, `Retriever`, `DocumentLoader`, `VectorStore` 等），支持链式调用（chaining）、工具集成、RAG（检索增强生成）、记忆管理（Memory）和代理决策（Agent Reasoning）。

以下是一个**典型、简洁、生产就绪风格的 LangChain v0.1.x（基于 `langchain-core`/`langchain-community` 的现代推荐写法）技术示例**，使用 Python 实现一个 RAG 流水线（含文档加载、分块、向量化、检索与 LLM 生成）：

```python
# pip install langchain-community langchain-openai chromadb tiktoken python-dotenv

import os
from typing import List

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 配置环境（推荐使用 .env）
os.environ["OPENAI_API_KEY"] = "sk-..."  # 或通过 dotenv.load_dotenv